In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error

# ---------------------------------------------------------
# Bước 1: Đọc dữ liệu đã qua tiền xử lý
# ---------------------------------------------------------
train_df = pd.read_csv('../data/processed/train.csv')
test_df = pd.read_csv('../data/processed/test.csv')

target_col = 'Price' 

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Test: X={X_test.shape}, y={y_test.shape}")

# ---------------------------------------------------------
# Bước 2: Huấn luyện mô hình Baseline (Linear Regression)
# ---------------------------------------------------------
print("\nMô hình Linear Regression")
model = LinearRegression()
model.fit(X_train, y_train)

# ---------------------------------------------------------
# Bước 3: Dự báo trên tập Test
# ---------------------------------------------------------
y_pred_log = model.predict(X_test)

# KHẮC PHỤC LỖI OVERFLOW (TRÀN SỐ)
y_pred_log_clipped = np.clip(y_pred_log, a_min=0, a_max=19)

# ---------------------------------------------------------
# Bước 4: INVERSE TRANSFORM (Biến đổi ngược từ Log về Tiền Thật)
# ---------------------------------------------------------
y_train_real = np.expm1(y_train) 
y_test_real = np.expm1(y_test)
y_pred_real = np.expm1(y_pred_log_clipped) 

# ---------------------------------------------------------
# Bước 5: Tính toán các Metrics dựa trên GIÁ TRỊ TIỀN THẬT
# ---------------------------------------------------------
mae = mean_absolute_error(y_test_real, y_pred_real)
mse = mean_squared_error(y_test_real, y_pred_real)
rmse = np.sqrt(mse) 
r2 = r2_score(y_test_real, y_pred_real)
mape = mean_absolute_percentage_error(y_test_real, y_pred_real)

metrics_data = {
    "Chỉ số đánh giá (Metric)": ["MAE", "RMSE", "R² Score", "MAPE"],
    "Giá trị (Value)": [f"{mae:,.2f}", f"{rmse:,.2f}", f"{r2:.4f}", f"{mape * 100:,.2f}%"]
}
metrics_df = pd.DataFrame(metrics_data)

print("\n--- BẢNG KẾT QUẢ ĐÁNH GIÁ BASELINE ---")
display(metrics_df) 

# ---------------------------------------------------------
# Bước 6: Phân tích MAE theo phân khúc giá tiền thật
# ---------------------------------------------------------
results_df = pd.DataFrame({
    'Actual_Price': y_test_real,
    'Predicted_Price': y_pred_real
})

def categorize_price(price):
    if price < 5000.0:
        return '1. Phân khúc Thấp (< 5 tỷ)'
    elif price <= 15000.0:
        return '2. Phân khúc Trung bình (5 - 15 tỷ)'
    else:
        return '3. Phân khúc Cao (> 15 tỷ)'

results_df['Segment'] = results_df['Actual_Price'].apply(categorize_price)

segment_results = []
grouped = results_df.groupby('Segment')

for segment, group in grouped:
    segment_mae = mean_absolute_error(group['Actual_Price'], group['Predicted_Price'])
    segment_results.append({
        "Phân khúc": segment,
        "Số lượng mẫu": f"{len(group):,}",
        "MAE": f"{segment_mae:,.2f}"
    })

segment_df = pd.DataFrame(segment_results)
print("\n--- BẢNG MAE THEO PHÂN KHÚC GIÁ ---")
display(segment_df)

Train: X=(37031, 59), y=(37031,)
Test: X=(9258, 59), y=(9258,)

Mô hình Linear Regression

--- BẢNG KẾT QUẢ ĐÁNH GIÁ BASELINE ---


,Chỉ số đánh giá (Metric),Giá trị (Value)
0,MAE,"13,145.64"
1,RMSE,"45,214.97"
2,R² Score,-0.0052
3,MAPE,356.80%



--- BẢNG MAE THEO PHÂN KHÚC GIÁ ---


,Phân khúc,Số lượng mẫu,MAE
0,1. Phân khúc Thấp (< 5 tỷ),"3,896","3,137.91"
1,2. Phân khúc Trung bình (5 - 15 tỷ),"3,225","4,172.64"
2,3. Phân khúc Cao (> 15 tỷ),"2,137","44,932.27"


## KIỂM TRA THÔNG SỐ

In [30]:
results_df = pd.DataFrame({
    "Actual": y_test_real,
    "Predicted": y_pred_real
})

results_df["Error"] = abs(
    results_df["Actual"] - results_df["Predicted"]
)

results_df.sort_values(
    by="Error",
    ascending=False
).head(20)

,Actual,Predicted,Error
8296,1582070.0,15417.122893,1.566653e+06
199,999000.0,12767.784852,9.862322e+05
3625,900000.0,10912.270986,8.890877e+05
6082,891000.0,5744.125028,8.852559e+05
1502,795000.0,15500.093440,7.794999e+05
6321,710000.0,11881.044834,6.981190e+05
322,680000.0,4554.461026,6.754455e+05
1342,630000.0,4991.382914,6.250086e+05
3348,630000.0,4991.382914,6.250086e+05
6171,498000.0,5124.489506,4.928755e+05


In [31]:
y_pred_log.min(), y_pred_log.max()

(np.float64(-4141.336900024936), np.float64(12.805970737883902))

In [32]:
train_df.describe()

,Area,Width,Length,Bedrooms,Bathrooms,Floors,Alley Width,Agent Listing Count,is_luxury,District_Huyện Bình Chánh,...,Direction_Unknown,Direction_Đông,Direction_Đông Bắc,Direction_Đông Nam,Road Type_Đường bê tông,Road Type_Đường nhựa,Road Type_Đường đá,Road Type_Đường đất,Agent Role_Môi giới,Price
count,3.703100e+04,3.703100e+04,3.703100e+04,3.703100e+04,3.703100e+04,3.703100e+04,3.703100e+04,3.703100e+04,37031.000000,37031.000000,...,37031.000000,37031.000000,37031.000000,37031.000000,37031.000000,37031.000000,37031.000000,37031.000000,37031.000000,37031.000000
mean,5.372579e-17,3.914307e-17,1.112891e-17,-9.593891e-18,2.225783e-17,6.360750e-17,6.427907e-17,3.856744e-17,0.457077,0.035214,...,0.781318,0.021441,0.019740,0.030623,0.030785,0.252113,0.007372,0.008128,0.930086,8.837531
std,1.000014e+00,1.000014e+00,1.000014e+00,1.000014e+00,1.000014e+00,1.000014e+00,1.000014e+00,1.000014e+00,0.498161,0.184322,...,0.413358,0.144853,0.139108,0.172296,0.172737,0.434232,0.085546,0.089791,0.255006,1.229221
min,-7.819332e-03,-5.396093e-02,-6.568843e-02,-9.026204e-01,-1.427526e+00,-5.962045e-03,-8.267646e-03,-4.760538e-01,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.064711
25%,-7.811786e-03,-3.502727e-02,-2.413418e-02,-1.510760e-01,-1.054293e-01,-5.192295e-03,-6.805464e-03,-4.646782e-01,0.000000,0.000000,...,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,8.006701
50%,-7.806227e-03,-2.934717e-02,-2.413418e-02,-1.510760e-01,-1.054293e-01,-5.192295e-03,-6.805464e-03,-3.747069e-01,0.000000,0.000000,...,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,8.729235
75%,-7.776444e-03,-1.609361e-02,-2.224535e-02,-1.510760e-01,-1.054293e-01,-5.192295e-03,-6.805464e-03,4.722735e-02,1.000000,0.000000,...,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000,9.581973
max,1.508012e+02,1.892826e+02,9.560354e+01,4.682045e+01,7.128779e+01,1.924318e+02,1.827644e+02,6.061859e+00,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,19.113828


In [33]:
import pandas as pd
train_df = pd.read_csv('../data/processed/train.csv')
print(train_df.columns.tolist())

['Area', 'Width', 'Length', 'Bedrooms', 'Bathrooms', 'Floors', 'Alley Width', 'Agent Listing Count', 'is_luxury', 'District_Huyện Bình Chánh', 'District_Huyện Cần Giờ', 'District_Huyện Củ Chi', 'District_Huyện Hóc Môn', 'District_Huyện Nhà Bè', 'District_Huyện Thanh Quan', 'District_Quận 1', 'District_Quận 10', 'District_Quận 11', 'District_Quận 12', 'District_Quận 2', 'District_Quận 3', 'District_Quận 4', 'District_Quận 5', 'District_Quận 6', 'District_Quận 7', 'District_Quận 8', 'District_Quận 9', 'District_Quận Bình Thạnh', 'District_Quận Bình Tân', 'District_Quận Gò Vấp', 'District_Quận Phú Nhuận', 'District_Quận Thủ Đức', 'District_Quận Thủ Đức (Cũ)', 'District_Quận Tân Bình', 'District_Quận Tân Phú', 'District_TP. Hồ Chí Minh', 'District_TP. Hồ Chí Minh(Mới)', 'District_Unknown', 'Property Type_Kho, nhà xưởng', 'Property Type_Khách sạn', 'Property Type_Nhà riêng', 'Property Type_Nhà trọ', 'Property Type_Văn phòng', 'Property Type_Đất', 'Position_Unknown', 'Position_Đường chính', 